In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import timedelta

# =========================================================
# ASSUMPTIONS
# =========================================================
# Existing DataFrames:
#
df_suppliers = pd.read_csv("data/raw/df_suppliers.csv")
df_po=pd.read_csv("data/raw/df_po.csv")
df_shipments=pd.read_csv("data/raw/df_shipments.csv")
df_quality_inspections= pd.read_csv("data/raw/df_quality_inspections.csv")
df_inventory   = pd.read_csv("data/raw/df_inventory.csv")
#
# =========================================================


audit_rows = []

audit_counter = 1


# =========================================================
# BUILD SUPPLIER PERFORMANCE METRICS
# =========================================================

# ---------------------------------------------------------
# Shipment delay metrics
# ---------------------------------------------------------

shipment_metrics = (

    df_shipments

    .groupby("supplier_id")

    .agg(

        total_shipments=(
            "shipment_id",
            "count"
        ),

        delayed_shipments=(
            "shipment_status",
            lambda x:
                (x == "Delayed").sum()
        )

    )

    .reset_index()

)

shipment_metrics["delay_ratio"] = (

    shipment_metrics["delayed_shipments"]

    /

    shipment_metrics["total_shipments"]

)


# ---------------------------------------------------------
# Quality metrics
# ---------------------------------------------------------

quality_metrics = (

    df_quality_inspections

    .groupby("supplier_id")

    .agg(

        total_inspected=(
            "inspected_units",
            "sum"
        ),

        total_rejected=(
            "rejected_units",
            "sum"
        )

    )

    .reset_index()

)

quality_metrics["rejection_ratio"] = (

    quality_metrics["total_rejected"]

    /

    quality_metrics["total_inspected"]

)

quality_metrics["rejection_ratio"] = (

    quality_metrics["rejection_ratio"]

    .fillna(0)

)


# ---------------------------------------------------------
# Inventory risk metrics
# ---------------------------------------------------------

inventory_metrics = (

    df_inventory

    .groupby("supplier_id")

    .apply(

        lambda x:
        (
            x["current_stock"]
            <
            x["safety_stock"]
        ).mean()

    )

    .reset_index(name="low_stock_ratio")

)


# =========================================================
# MERGE SUPPLIER RISK PROFILE
# =========================================================

supplier_risk = (

    df_suppliers[[
        "supplier_id",
        "criticality_level",
        "country"
    ]]

    .merge(
        shipment_metrics,
        on="supplier_id",
        how="left"
    )

    .merge(
        quality_metrics,
        on="supplier_id",
        how="left"
    )

    .merge(
        inventory_metrics,
        on="supplier_id",
        how="left"
    )

)


supplier_risk = supplier_risk.fillna(0)


# =========================================================
# GENERATE AUDITS
# =========================================================

for _, supplier in supplier_risk.iterrows():

    supplier_id = supplier["supplier_id"]

    criticality = supplier["criticality_level"]


    # -----------------------------------------------------
    # AUDIT FREQUENCY
    # -----------------------------------------------------

    audit_count = {

        1: 1,
        2: 1,
        3: 2,
        4: 3,
        5: 4

    }[criticality]


    for _ in range(audit_count):

        # -------------------------------------------------
        # BASE SCORE
        # -------------------------------------------------

        audit_score = 100


        # -------------------------------------------------
        # DELAY PENALTY
        # -------------------------------------------------

        audit_score -= (

            supplier["delay_ratio"]
            * 25
        )


        # -------------------------------------------------
        # QUALITY PENALTY
        # -------------------------------------------------

        audit_score -= (

            supplier["rejection_ratio"]
            * 100
        )


        # -------------------------------------------------
        # INVENTORY RISK PENALTY
        # -------------------------------------------------

        audit_score -= (

            supplier["low_stock_ratio"]
            * 20
        )


        # -------------------------------------------------
        # INTERNATIONAL RISK
        # -------------------------------------------------

        if supplier["country"] != "India":

            audit_score -= np.random.uniform(1, 5)


        # -------------------------------------------------
        # RANDOM AUDIT VARIANCE
        # -------------------------------------------------

        audit_score += np.random.normal(
            0,
            3
        )


        # -------------------------------------------------
        # LIMIT SCORE
        # -------------------------------------------------

        audit_score = max(
            min(round(audit_score, 2), 100),
            35
        )


        # -------------------------------------------------
        # MAJOR FINDINGS
        # -------------------------------------------------

        major_findings = 0

        if audit_score < 85:

            major_findings += np.random.randint(0, 2)

        if audit_score < 70:

            major_findings += np.random.randint(1, 4)

        if supplier["rejection_ratio"] > 0.08:

            major_findings += 1


        # -------------------------------------------------
        # MINOR FINDINGS
        # -------------------------------------------------

        minor_findings = np.random.randint(0, 6)

        if supplier["delay_ratio"] > 0.20:

            minor_findings += np.random.randint(1, 4)


        # -------------------------------------------------
        # COMPLIANCE STATUS
        # -------------------------------------------------

        if audit_score >= 90:

            compliance_status = "Compliant"

        elif audit_score >= 75:

            compliance_status = "Conditional"

        elif audit_score >= 60:

            compliance_status = "Under Review"

        else:

            compliance_status = "Non-Compliant"


        # -------------------------------------------------
        # AUDIT DATE
        # -------------------------------------------------

        audit_date = (

            pd.Timestamp("2025-01-01")

            +

            timedelta(
                days=np.random.randint(0, 365)
            )

        )


        # -------------------------------------------------
        # CREATE AUDIT RECORD
        # -------------------------------------------------

        audit_rows.append({

            "audit_id":
                f"AUD{audit_counter:07}",

            "supplier_id":
                supplier_id,

            "audit_date":
                audit_date.date(),

            "audit_score":
                audit_score,

            "major_findings":
                major_findings,

            "minor_findings":
                minor_findings,

            "compliance_status":
                compliance_status
        })


        audit_counter += 1


# =========================================================
# FINAL AUDIT TABLE
# =========================================================

df_supplier_audits = pd.DataFrame(
    audit_rows
)


# =========================================================
# OPTIONAL VALIDATIONS
# =========================================================

print("\n==============================")
print("AUDIT SUMMARY")
print("==============================")

print(
    f"Total Audits: "
    f"{len(df_supplier_audits)}"
)

print("\nCompliance Distribution:")

print(
    df_supplier_audits[
        "compliance_status"
    ].value_counts()
)

print("\nAudit Score Summary:")

print(
    df_supplier_audits[
        "audit_score"
    ].describe()
)

print("\nSample Records:\n")

print(
    df_supplier_audits.head()
)

In [ ]:
df_supplier_audits.to_csv("data/raw/df_supplier_audits.csv", index=False)